1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## Подготовка

## 1. Импорт библиотек

In [1]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

## 2. Скачивание и распаковка датасета

In [2]:
import gdown

gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip',
    None,
    quiet=True
)

!unzip -o writers.zip -d writers/

Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


## 3. Определение гиперпараметров (VOCAB_SIZE, WIN_SIZE, WIN_HOP, BATCH_SIZE, EPOCHS, EMBED_SIZE)

In [3]:
FILE_DIR = 'writers'

VOCAB_SIZE = 20000
WIN_SIZE = 200
WIN_HOP = 50

BATCH_SIZE = 64
EPOCHS = 5
EMBED_SIZE = 64

## 4. Загрузка файлов и создание списка текстов

In [4]:
CLASS_LIST = []
texts = []

for file in os.listdir(FILE_DIR):

    if file.endswith('.txt'):

        author = file.replace('.txt', '')
        CLASS_LIST.append(author)

        with open(os.path.join(FILE_DIR, file), 'r', encoding='utf-8') as f:
            texts.append(f.read().replace('\n', ' '))

## 5. Построение словаря (build_vocab)

In [5]:
def build_vocab(texts, vocab_size):

    words = []

    for t in texts:
        words.extend(t.split())

    counter = Counter(words)

    most_common = counter.most_common(vocab_size - 1)

    vocab = {w:i+1 for i,(w,_) in enumerate(most_common)}

    return vocab


vocab = build_vocab(texts, VOCAB_SIZE)

## 6. Кодирование текстов в числа

In [6]:
def encode(text, vocab):
    return [vocab.get(w, 0) for w in text.split()]

encoded = [encode(t, vocab) for t in texts]

## 7. Нарезка датасета на окна (create_dataset)

In [7]:
def create_dataset(data, win_size, hop):

    X, y = [], []

    for class_id, seq in enumerate(data):

        for i in range(0, len(seq) - win_size, hop):

            X.append(seq[i:i+win_size])
            y.append(class_id)

    return np.array(X), np.array(y)


X, y = create_dataset(encoded, WIN_SIZE, WIN_HOP)

print(X.shape)

(50698, 200)


## 8. Создание Dataset и DataLoader (разделение на train/test)

In [8]:
class TextDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = TextDataset(X, y)

train_size = int(len(dataset)*0.8)
test_size = len(dataset) - train_size

train_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

## 9. Определение архитектуры модели

In [20]:
class Model(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.emb = nn.Embedding(vocab_size, EMBED_SIZE)

        self.gru = nn.GRU(EMBED_SIZE, 64, batch_first=True)

        self.fc = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, len(CLASS_LIST))
        )

    def forward(self, x):

        x = self.emb(x)

        _, h = self.gru(x)

        return self.fc(h[-1])

## 10. Цикл обучения (train loop)

In [12]:
for epoch in range(EPOCHS):

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        out = model(X_batch)

        loss = loss_fn(out, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(out, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    acc = correct / total

    print(f"Epoch {epoch+1} | Loss: {total_loss:.2f} | Train Acc: {acc:.4f}")

Epoch 1 | Loss: 56.97 | Train Acc: 0.9711
Epoch 2 | Loss: 25.44 | Train Acc: 0.9881
Epoch 3 | Loss: 10.74 | Train Acc: 0.9960
Epoch 4 | Loss: 7.45 | Train Acc: 0.9971
Epoch 5 | Loss: 5.80 | Train Acc: 0.9977


## 11. Оценка accuracy на тестовой выборке

In [13]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        out = model(X_batch)

        pred = torch.argmax(out, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

print("TEST ACC:", correct / total)

TEST ACC: 0.9536489151873767


## 12. Сохранение предсказаний для каждого батча

In [15]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        out = model(X_batch)

        preds = torch.argmax(out, dim=1)

        y_true.extend(y_batch.numpy())
        y_pred.extend(preds.numpy())

## 13. Демонстрация предсказаний по каждому классу

In [16]:
correct = 0
shown = set()

for i in range(len(y_true)):

    if y_true[i] not in shown:

        shown.add(y_true[i])

        print("Истинный класс :", CLASS_LIST[y_true[i]])
        print("Предсказанный   :", CLASS_LIST[y_pred[i]])
        print()

        if y_true[i] == y_pred[i]:
            correct += 1

print("Правильно распознано классов:", correct, "из", len(CLASS_LIST))

Истинный класс : (Рэй Брэдберри) Обучающая_22 вместе
Предсказанный   : (Рэй Брэдберри) Обучающая_22 вместе

Истинный класс : (Макс Фрай) Обучающая_5 вместе
Предсказанный   : (Макс Фрай) Обучающая_5 вместе

Истинный класс : (Клиффорд_Саймак) Обучающая_5 вместе
Предсказанный   : (Клиффорд_Саймак) Обучающая_5 вместе

Истинный класс : (О. Генри) Тестовая_20 вместе
Предсказанный   : (Клиффорд_Саймак) Обучающая_5 вместе

Истинный класс : (О. Генри) Обучающая_50 вместе
Предсказанный   : (О. Генри) Обучающая_50 вместе

Истинный класс : (Стругацкие) Обучающая_5 вместе
Предсказанный   : (Стругацкие) Обучающая_5 вместе

Истинный класс : (Клиффорд_Саймак) Тестовая_2 вместе
Предсказанный   : (Клиффорд_Саймак) Тестовая_2 вместе

Истинный класс : (Макс Фрай) Тестовая_2 вместе
Предсказанный   : (Макс Фрай) Тестовая_2 вместе

Истинный класс : (Булгаков) Обучающая_5 вместе
Предсказанный   : (Булгаков) Обучающая_5 вместе

Истинный класс : (Булгаков) Тестовая_2 вместе
Предсказанный   : (Булгаков) Тестовая